# Phase 2 — Learned detector (Colab, GPU runtime)

Trains the frozen-DINOv2 + attention-pooling detector and wires it into the
fusion pipeline. All logic lives in repo modules (`training/`, `ai_image_id/detector/`);
this notebook only orchestrates. **Runtime → Change runtime type → GPU** for step 2.

In [ ]:
# Setup (same as phase 1 notebook)
!apt-get -qq install -y libimage-exiftool-perl > /dev/null
# Option A: upload + unzip the repo zip; Option B: git clone your repo
# !git clone https://github.com/<you>/ai-image-id.git ai-image-id-mvp
%pip install -q -e ./ai-image-id-mvp
%pip install -q torch --index-url https://download.pytorch.org/whl/cu121
import sys; sys.path.insert(0, "./ai-image-id-mvp")  # exposes training/ package

## Step 1 — Data: GenImage subset with matched confounds

Download a GenImage subset (e.g. the SDv1.4 generator split + ImageNet reals —
see https://github.com/GenImage-Dataset/GenImage for links). Then build
de-confounded train/val splits: identical resize + shared JPEG-Q distribution
for BOTH classes, so the head can't learn compression shortcuts.

In [ ]:
from training.prepare_data import prepare_split

# adjust paths to wherever you unpacked GenImage in Drive/Colab
prepare_split("genimage/nature", "genimage/sdv14/fake", "data/train",
              n_per_class=20_000, seed=0)
prepare_split("genimage/nature_val", "genimage/sdv14/fake_val", "data/val",
              n_per_class=2_000, seed=1)

## Step 2 — Precompute frozen embeddings (GPU, one pass)

The expensive step, done once. `n_aug=2` adds two robustness-augmented variants
(JPEG 30–90 / resize / blur) per image so the head trains on transformed inputs
without re-running the backbone. ~GPU-hours depending on subset size; shards go
to Drive if you point out_dir there.

In [ ]:
from training.embed import precompute

precompute("data/train/manifest.csv", "data/train", "emb/train", n_aug=2, device="cuda")
precompute("data/val/manifest.csv",   "data/val",   "emb/val",   n_aug=0, device="cuda")

## Step 3 — Train the head (minutes, CPU is fine)

In [ ]:
from training.train_head import train

ckpt = train("emb/train", "emb/val", out="head.pt", epochs=5, device="cuda")

## Step 4 — Calibrate (temperature scaling) and report ECE

In [ ]:
from training.calibrate_eval import calibrate

report = calibrate("head.val_logits.npz", out_json="head.calibration.json")
print(report)  # temperature, ECE before/after

## Step 5 — Cross-generator evaluation

Precompute embeddings for held-out generators (Midjourney, ADM, BigGAN, GLIDE…
from GenImage) and build the honest table: expect strong scores near your
training generator and degradation on modern commercial ones — that's the
finding that motivates fusion.

In [ ]:
import numpy as np, torch
from training.train_head import build_head, _load_shards
from training.calibrate_eval import cross_generator_table, _sigmoid
import json

state = torch.load("head.pt", map_location="cpu")
head = build_head(state["dim"]); head.load_state_dict(state["state_dict"]); head.eval()
T = json.load(open("head.calibration.json"))["temperature"]

def probs_for(emb_dir):
    ps, ys = [], []
    for patch, cls, y in _load_shards(emb_dir):
        with torch.no_grad():
            logits = head(torch.from_numpy(patch), torch.from_numpy(cls)).numpy()
        ps.append(_sigmoid(logits / T)); ys.append(y)
    return np.concatenate(ps), np.concatenate(ys).astype(np.float64)

results = {g: probs_for(f"emb/test_{g}") for g in ["midjourney", "adm", "biggan", "glide"]}
print(cross_generator_table(results))

## Step 6 — Use it in the pipeline

Point the pipeline at the checkpoint (arg or `AI_IMAGE_ID_HEAD` env var). The
fusion engine now uses the calibrated score — capped at `likely`, drift-aware,
and able to output `unlikely` for confidently-camera images.

In [ ]:
from ai_image_id.main import analyze_image

result = analyze_image("some_image.jpg", detector_ckpt="head.pt")
print(result.ai_verdict, result.confidence)
print(result.evidence.detector)
print(result.notes)